In [43]:
import re
import joblib
import numpy as np
import pandas as pd
import nltk
import ftfy
import os

import os
from dotenv import load_dotenv
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from scipy.sparse import hstack, csr_matrix


In [44]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rivin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rivin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\rivin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

# Emotion Lexicons

In [45]:
# These are predefined word sets which will be used to identify the specific emotional states in text
JOY_WORDS = {
    'happy','happiness','joy','joyful','excited','excitement','amazing','wonderful',
    'fantastic','great','love','loved','grateful','gratitude','blessed','thrilled',
    'delighted','ecstatic','overjoyed','proud','celebrate','celebrating','smiling',
    'laughing','laugh','beautiful','incredible','awesome','lucky','thankful',
    'promoted','promotion','scholarship','accepted','engaged','proposal','born',
    'baby','married','wedding','vacation','holiday','trip','surprise','gift',
    'moon','unstoppable','alive','peace','peaceful','magical','crossed','finish',
    'line','milestone','achievement','achieved','dream','worth','penny','saved',
    'cheering','dancing','danced','winning','won','passed','congratulations',
    'glowing','beaming','radiant','light','smile','grinning','grin','elated',
    'content','fulfilling','fulfilled','meaningful','rewarding','rewarded'
}

ANXIETY_WORDS = {
    'anxious','anxiety','worry','worried','worrying','panic','panicking','fear',
    'fearful','terrified','terrifying','terror','dread','dreading','nervous',
    'catastrophize','catastrophizing','overthink','overthinking','racing',
    'trembling','shaking','freeze','frozen','avoid','avoidance','scared',
    'breathe','chest','tightens','tighten','attacks','attack','scenarios',
    'convinced','wrong','shake','judged','highways','losing','control',
    'smaller','triggered','trigger','worrying','terrible','cancelled',
    'interviews','crowd','crowded','spiral','spiraling','restless','uneasy',
    'apprehensive','tense','tension','phobia','obsess','obsessing',
    'checking','locks','ruminate','ruminating','sleepless','insomnia',
    'heart','pounding','sweating','sweat','nauseous','nausea','faint'
}

STRESS_WORDS = {
    'stress','stressed','stressful','burnout','burned','overwhelmed','overloaded',
    'exhausted','exhaustion','pressure','deadline','workload','furious','angry',
    'anger','irritable','irritated','snapping','frustrated','frustration',
    'relentless','piling','humiliated','juggling','rent','cover','barely',
    'syllabus','missed','launch','client','sixteen','edge','temper','aches',
    'breaking','crushing','meetings','running','empty','behind','overdue',
    'manager','boss','colleague','fired','layoff','debt','bills','financial',
    'screaming','yelling','yelled','slammed','snapped','boiling','fuming',
    'seething','livid','outraged','grinding','drained','depleted','stretched',
    'pulled','pushed','collapsing','crumbling','falling','apart','cope','coping'
}

DEPRESSION_WORDS = {
    'depressed','depression','hopeless','hopelessness','empty','numb','worthless',
    'useless','lonely','loneliness','isolated','isolation','crying','cried',
    'sadness','sad','miserable','darkness','dark','pointless','meaningless',
    'void','hollow','broken','shattered','defeated','lost','invisible',
    'disconnected','ghost','drifting','faking','fake','mask','pretend',
    'exhausted','heavy','sinking','drowning','suffocating','trapped','stuck',
    'failure','failed','disappoint','disappointing','ashamed','shame','guilt',
    'regret','nothingness','bleak','grey','gray','colorless','lifeless',
    'apathy','apathetic','withdrawn','detached','unmotivated','unmoved'
}

SUICIDAL_WORDS = {
    'suicidal','suicide','die','dying','dead','death','kill','goodbye','note',
    'plan','end','bridge','pills','overdose','cut','cutting','harm','pain',
    'method','ready','disappear','disappeared','letgo','peace','relieved',
    'farewell','arranged','neighbor','dog','stopping','almost',
    'tonight','night','stockpiling','researched','methods','reason','stay',
    'birthday','unbearable','terminal','decided','terms','deeper','wakeup',
    'morning','hoped','awake','knife','rope','jump','jumped','ledge','final'
}

NEGATION_WORDS = {
    'not','never','no','nothing','nobody','nowhere','neither','nor','none',
    'cannot','cant','wont','wouldnt','shouldnt','couldnt','dont','doesnt',
    'didnt','hasnt','havent','hadnt','isnt','arent','wasnt','werent'
}


# Text cleaning functions

In [46]:
lemmatizer = WordNetLemmatizer()

def fix_encoding(text):
    if not isinstance(text, str):
        return text
    text = ftfy.fix_text(text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def remove_patterns(text):
    text = str(text).lower()
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\[.*?\]\(.*?\)', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)

def lemmatize_text(text):
    tokens = word_tokenize(text)
    result = []
    for token in tokens:
        if not token.isalpha():
            continue
        pos   = get_wordnet_pos(token)
        lemma = lemmatizer.lemmatize(token, pos)
        result.append(lemma)
    return ' '.join(result)

# Feature Engineering

In [47]:
def count_lexicon(text, lexicon):
    tokens = set(text.lower().split())
    return len(tokens & lexicon)

def has_negation(text):
    tokens = set(text.lower().split())
    return int(bool(tokens & NEGATION_WORDS))

def exclamation_count(text):
    return text.count('!')

def question_count(text):
    return text.count('?')

def caps_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if c.isupper()) / len(letters)

def avg_word_length(text):
    words = text.split()
    if not words:
        return 0.0
    return sum(len(w) for w in words) / len(words)

def build_features(df):
    features = pd.DataFrame()
    features['num_chars']  = df['text'].str.len()
    features['num_sentences'] = df['text'].apply(lambda x: len(sent_tokenize(str(x))))
    features['num_words'] = df['text'].str.split().str.len()
    features['avg_word_len'] = df['text'].apply(avg_word_length)
    features['exclamation'] = df['text'].apply(exclamation_count)
    features['question']  = df['text'].apply(question_count)
    features['caps_ratio']  = df['text'].apply(caps_ratio)
    features['has_negation'] = df['text'].apply(has_negation)
    features['joy_score']  = df['text'].apply(lambda x: count_lexicon(x, JOY_WORDS))
    features['anxiety_score'] = df['text'].apply(lambda x: count_lexicon(x, ANXIETY_WORDS))
    features['stress_score'] = df['text'].apply(lambda x: count_lexicon(x, STRESS_WORDS))
    features['depression_score'] = df['text'].apply(lambda x: count_lexicon(x, DEPRESSION_WORDS))
    features['suicidal_score'] = df['text'].apply(lambda x: count_lexicon(x, SUICIDAL_WORDS))
    return features.values

# load Models

In [48]:
MODEL_DIR = '../models/'

lr_model = joblib.load(os.path.join(MODEL_DIR, 'lr_model.pkl'))
bnb_model = joblib.load(os.path.join(MODEL_DIR, 'bnb_model.pkl'))
xgb_model = joblib.load(os.path.join(MODEL_DIR, 'xgb_model.pkl'))

# vectorizers for text processing
vectorizers = joblib.load(os.path.join(MODEL_DIR, 'vectorizer.pkl'))

# label encoder for class mapping
le = joblib.load(os.path.join(MODEL_DIR, 'label_encoder.pkl'))

# unigram and bigram tfidf
tfidf_uni = vectorizers['unigram']
tfidf_bi = vectorizers['bigram']

# All class names in the order sklearn assigned them in alphabetical
CLASSES = list(le.classes_)

print("Models loaded")
print("\nLabel mapping")
for idx, label in enumerate(le.classes_):
    print(f"{idx} to {label}")

Models loaded

Label mapping
0 to Anxiety
1 to Depression
2 to Normal
3 to Stress
4 to Suicidal


# Model Weights

In [49]:
# One weight per model applied
# Based on overall training accuracy LR with highest weight , XGB boost 2nd while BNB weakest
MODEL_WEIGHTS = {
    'lr' : 0.50,
    'bnb': 0.20,
    'xgb': 0.30,
}

print("Model weights")
for model, w in MODEL_WEIGHTS.items():
    print(f"{model.upper()} gets {w}")

Model weights
LR gets 0.5
BNB gets 0.2
XGB gets 0.3


# Combined Model Voting Prediction

In [ ]:
def preprocess(text: str):
    clean = lemmatize_text(remove_patterns(fix_encoding(text)))
    row = pd.DataFrame([{'text': text}])
    uni = tfidf_uni.transform([clean])
    bi = tfidf_bi.transform([clean])
    num = csr_matrix(build_features(row))
    return hstack([uni, bi, num])


# Runs all 3 models and combines their results using weights and returns individual predictions and final result
def predict_label(text: str) -> dict:

    X = preprocess(text)

    # Logistic Regression gives probabilities for all classes
    lr_proba = lr_model.predict_proba(X)[0]

    # XGBoost also gives probabilities
    xgb_proba = xgb_model.predict_proba(X)[0]

    # But Naive Bayes only gives one class, so convert to one hot for example if sucidal it's going to be [0, 0, 0, 0, 1]
    bnb_raw = bnb_model.predict(X)[0]
    bnb_vote = np.zeros(len(CLASSES))
    bnb_vote[bnb_raw] = 1.0

    # Combine predictions from all models using weighted voting
    combined = (MODEL_WEIGHTS['lr']  * lr_proba  + MODEL_WEIGHTS['bnb'] * bnb_vote  + MODEL_WEIGHTS['xgb'] * xgb_proba)

    # Get the final prediction where highest score wins
    final_idx = np.argmax(combined)
    final_label = le.inverse_transform([final_idx])[0]

    return {
        'lr_pred'    : le.inverse_transform([np.argmax(lr_proba)])[0],
        'bnb_pred'   : le.inverse_transform([bnb_raw])[0],
        'xgb_pred'   : le.inverse_transform([np.argmax(xgb_proba)])[0],
        'final_label': final_label,
        'scores'     : dict(zip(CLASSES, combined.round(4)))
    }

# Crisis Keyword Routing System

In [51]:
# Crisis keywords that override the model for safety
KEYWORD_CRISIS = {
    'kill myself', 'end my life', 'want to die', 'going to kill',
    'suicide', 'overdose', 'goodbye forever', 'end it all',
    'no reason to live', 'better off dead', 'cant go on',
    'take my own life', 'dont want to be here', 'ready to die',
    'life isnt worth', 'nothing left to live for', 'final goodbye',
    'jumping off', 'hanging myself', 'cut my wrists', 'slit my',
    'self harm', 'hurt myself badly', 'harm myself',
    'want it to end', 'ready to go', 'cant take it anymore',
    'everyone better without me', 'burden to everyone',
    'world without me', 'stockpiling pills', 'writing goodbye',
    'suicide note', 'plan to end', 'wont be here tomorrow',
    'final message', 'kill me', 'wish i was dead',
    'shouldnt exist', 'disappear forever', 'cease to exist'
}

def analyze_safety(user_text: str) -> dict:
    lower = user_text.lower()
    keyword_hit = any(kw in lower for kw in KEYWORD_CRISIS)

    # Run the emotion model for every message so we always keep the user's emotional signal
    model_detail = predict_label(user_text)
    model_label = model_detail['final_label']
    model_crisis = (model_label == 'Suicidal')

    safety_flag = 'crisis' if (keyword_hit or model_crisis) else 'non_crisis'

    if keyword_hit and model_crisis:
        triggered_by = 'both'
    elif keyword_hit:
        triggered_by = 'keyword'
    elif model_crisis:
        triggered_by = 'model'
    else:
        triggered_by = 'none'

    return {
        'user_message': user_text,
        'emotion_label': model_label,
        'safety_flag': safety_flag,
        'triggered_by': triggered_by,
        'keyword_crisis_match': keyword_hit,
        'model_scores': model_detail.get('scores', {}),
        'model_detail': model_detail
    }

# Testing

In [65]:
user_message = "Today was a really hot day i wish i ate an ice cream"
result = analyze_safety(user_message)

# Only ouput message , crisis or not and label predicted
output = {
    'message': result['user_message'],
    'Emotion Label': result['emotion_label'],
    }

print(output)

{'message': 'Today was a really hot day i wish i ate an ice cream', 'Emotion Label': 'Normal'}


# Gemini API Setup and Configuration

In [53]:
# Load environment variables
load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")

genai.configure(api_key=GEMINI_API_KEY)

# Initialize Chat Memory and User Profile

In [54]:
conversation_history = []

user_profile = {
    'summary': 'New user with no conversation history yet.',
    'dominant_emotions': [],
    'crisis_count': 0,
    'total_messages': 0
}

print(f"User profile: {user_profile}")

User profile: {'summary': 'New user with no conversation history yet.', 'dominant_emotions': [], 'crisis_count': 0, 'total_messages': 0}


# Setup Folders for Data and Database

In [55]:
RAG_DOCS_DIR = '../data/rag_documents'
CHROMA_DB_DIR = '../data/chroma_db'
os.makedirs(RAG_DOCS_DIR, exist_ok=True)
os.makedirs(CHROMA_DB_DIR, exist_ok=True)

# Initialize AI Models and Settings

In [56]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=GEMINI_API_KEY,
    temperature=0.7,
    max_output_tokens=1024
)

response = llm.invoke("Hello how are you")

print(response.text)

I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?


# Cancer Knowledge Base Loading and Chunking

In [57]:
with open('../data/cancer_knowledge_base.json', 'r', encoding='utf-8') as f:
    knowledge_data = json.load(f)

documents = []
# Convert each json entry into a langchain doc
for item in knowledge_data:
    doc = Document(
        page_content=item['content'],
        metadata={
            'id': item['id'],
            'category': item['category'],
            'title': item.get('title', ''),
            'region': item.get('region', 'General'),
            'keywords': ', '.join(item.get('keywords', []))
        }
    )
    documents.append(doc)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50, # Avoid losing meaning at chunk boundaries
    length_function=len, # measures chunk size 
    separators=["\n\n", "\n", ". ", " ", ""] # Split by paragraphs first, then lines, then sentences, then words, and finally characters
)

split_docs = text_splitter.split_documents(documents)

print(f"Original documents has {len(documents)}")
print(f"After chunking now {len(split_docs)} chunks")

Original documents has 1000
After chunking now 1998 chunks


# Vector Store Creation using Chroma DB

In [58]:
collection_name = "cancer_knowledge"

# Reset existing collection so reruns do not duplicate vectors
try:
    old_store = Chroma(
        collection_name=collection_name,
        persist_directory=CHROMA_DB_DIR,
        embedding_function=embeddings
    )
    old_store.delete_collection()
    print("Deleted existing collection and making fresh index")
except Exception:
    print("No previous collection found. Building new index")

# Chroma using embeddings
vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory=CHROMA_DB_DIR,
    collection_name=collection_name
)

print(f"Total chunks indexed are {len(split_docs)}")

Deleted existing collection and making fresh index


c:\Users\rivin\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Total chunks indexed are 1998


# Semantic Search and Manual Keyword Verification

In [59]:
# Common words to ignore 
DEFAULT_STOP_WORDS = {
    "where", "is", "the", "a", "an", "of", "to", "in", "for", "on", "at",
    "what", "which", "who", "when", "why", "how", "are", "was", "were", "be"
}

def extract_query_terms(query: str) -> set:
    # Extract important words from the user query 
    words = re.findall(r"\w+", query.lower())
    meaningful_terms = set()

    for t in words:
        if t not in DEFAULT_STOP_WORDS and len(t) > 2:
            meaningful_terms.add(t)

    return meaningful_terms

def doc_key(doc):
    # Create a unique key for each document inorder to used to avoid duplicates
    return doc.metadata.get("id") or doc.metadata.get("title") or doc.page_content[:120]

def lexical_score(doc, query_terms: set) -> int:
    # Merge title and content into one lowercase string for easy matching
    text = f"{doc.metadata.get('title', '')} {doc.page_content}".lower()
    
    # Break the text into individual words and remove any duplicates
    terms_in_doc = set(re.findall(r"\w+", text))
    overlap = query_terms & terms_in_doc # Using set intersection find words that exist in both groups

    return len(overlap)

In [ ]:
def search_knowledge(query: str, top_n: int = 2):
    query_terms = extract_query_terms(query)

    # Use vector search to find documents with a similar meaning to the question
    semantic_hits = vectorstore.similarity_search_with_relevance_scores(query, k=20)
    combined = {}

    for doc, sem_score in semantic_hits:
        key = doc_key(doc)
        combined[key] = {"doc": doc,"semantic": float(sem_score),
            "lexical": lexical_score(doc, query_terms),
        }

    # Manually check with keyword matching
    for doc in split_docs:
        key = doc_key(doc)
        lex= lexical_score(doc, query_terms)

        if lex <= 0:
            continue

        if key not in combined:
            combined[key] = {"doc": doc, "semantic": 0.0, "lexical": lex}
        else:
            combined[key]["lexical"] = max(combined[key]["lexical"], lex)

    # Use a 0.25 weight so keywords help rank the results without ignoring meaning
    ranked = sorted(
        combined.values(),
        key=lambda r: r["semantic"] + 0.25 * r["lexical"],
        reverse=True,
    )
    high_confidence = []
    for r in ranked:
        if r["semantic"] >= 0.30 or r["lexical"] >= 2:
            high_confidence.append(r)

    final_results = []
    seen_titles = set()

    for row in high_confidence:
        doc = row["doc"]
        title = doc.metadata.get("title") or "Untitled"
        title = title.strip()

        if title in seen_titles:
            continue

        seen_titles.add(title)
        final_results.append(row)

        if len(final_results) == top_n:
            break

    return final_results

def retrieve_knowledge(query: str, top_n: int = 2):
    results = search_knowledge(query, top_n=top_n)

    if results:
        contexts = []
        sources = []

        for row in results:
            doc = row["doc"]
            title = doc.metadata.get("title") or "Untitled"
            contexts.append(f"Source: {title}\n{doc.page_content}")
            sources.append(title)

        knowledge_context = "\n\n".join(contexts)
        return {
            "user_message": query,
            "knowledge_found": True,
            "knowledge_context": knowledge_context,
            "sources": sources,
            # Final prompt sent to LLM query and retrieved knowledge
            "prompt_input": f"User message:\n{query}\n\nKnowledge context:\n{knowledge_context}"
        }

    # If nothing found send query only
    return {
        "user_message": query,
        "knowledge_found": False,
        "knowledge_context": "",
        "sources": [],
        "prompt_input": query
    }

In [86]:
def formulate_prompt(safety_report: dict, knowledge_context: dict | None = None):
    # If the Sentinel detects a crisis we immediately force a safety first response and ignore any medical knowledge retrieval to avoid confusion
    if safety_report['safety_flag'] == 'crisis':
        return {
            "user_message": safety_report['user_message'],
            "sentiment_label": safety_report['emotion_label'],
            "safety_flag": safety_report['safety_flag'],
            "knowledge_found": False,
            "knowledge_context": "",
            "sources": [],
            "grounding_mode": "crisis_protocol",
            "response_style": "calm_safety_first",
            "prompt_input": safety_report['user_message']
        }
    # Making sure the function does not crash if no knowledge context is provided
    knowledge_context = knowledge_context or {
        "knowledge_found": False,
        "knowledge_context": "",
        "sources": [],
        "prompt_input": safety_report['user_message']
    }

    # Maps the ML model sentiment prediction to a conversational  tone
    style_map = {
        "Joy": "warm_positive",
        "Stress": "calm_structured",
        "Anxiety": "calm_reassuring",
        "Depression": "gentle_supportive",
        "Suicidal": "calm_safety_first"
    }
    response_style = style_map.get(safety_report['emotion_label'], "calm_supportive")

    # Instruction given to for the LLM
    return {
        "user_message": safety_report['user_message'],
        "sentiment_label": safety_report['emotion_label'],
        "safety_flag": safety_report['safety_flag'],
        "knowledge_found": knowledge_context['knowledge_found'],
        "knowledge_context": knowledge_context['knowledge_context'],
        "sources": knowledge_context['sources'],
        "grounding_mode": "strict_kb" if knowledge_context['knowledge_found'] else "general_assist", # Determines if the AI should stick strictly to our data or use its general knowledge
        "response_style": response_style,
        "prompt_input": knowledge_context['prompt_input']
    }

def run_pipeline(user_message: str, top_n: int = 2) -> dict:
    # Run the user message through the sentiment models and crisis keywords first
    safety_report = analyze_safety(user_message)

    # crisis messages do not go through retrieval
    if safety_report['safety_flag'] == 'crisis':
        knowledge_context = {
            "user_message": user_message,
            "knowledge_found": False,
            "knowledge_context": "",
            "sources": [],
            "prompt_input": user_message,
            "retrieval_skipped": True
        }
    else:
        # Fetch relevant chunks from ChromaDB using semantic and lexical search
        knowledge_context = retrieve_knowledge(user_message, top_n=top_n)
        knowledge_context['retrieval_skipped'] = False

    # Combine the safety and knowledge reports into the final instruction
    persona_packet = formulate_prompt(safety_report, knowledge_context)

    return {
        "safety": safety_report,
        "knowledge": knowledge_context,
        "persona": persona_packet
    }

In [87]:
user_message = "Where is Apeksha Hospital located?"
pipeline_output = run_pipeline(user_message, top_n=2)

print(json.dumps({
    "safety": {
        "emotion_label": pipeline_output["safety"]["emotion_label"],
        "safety_flag": pipeline_output["safety"]["safety_flag"],
        "triggered_by": pipeline_output["safety"]["triggered_by"]
    },
    "knowledge": {
        "knowledge_found": pipeline_output["knowledge"]["knowledge_found"],
        "sources": pipeline_output["knowledge"]["sources"],
        "retrieval_skipped": pipeline_output["knowledge"]["retrieval_skipped"]
    },
    "persona_prompt_input_preview": pipeline_output["persona"]["prompt_input"][:350]}, indent=2, ensure_ascii=False))

c:\Users\rivin\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{
  "safety": {
    "emotion_label": "Normal",
    "safety_flag": "non_crisis",
    "triggered_by": "none"
  },
  "knowledge": {
    "knowledge_found": true,
    "sources": [
      "Apeksha Hospital Maharagama"
    ],
    "retrieval_skipped": false
  },
  "persona_prompt_input_preview": "User message:\nWhere is Apeksha Hospital located?\n\nKnowledge context:\nSource: Apeksha Hospital Maharagama\nApeksha Hospital is Sri Lanka's largest cancer treatment center located in Maharagama. They provide free chemotherapy, radiotherapy, and palliative care services. The hospital operates under the Ministry of Health and all treatment is free for S"
}


In [66]:
user_message = "I want to hang myself"
pipeline_output = run_pipeline(user_message, top_n=2)

print(json.dumps({
    "safety": {
        "emotion_label": pipeline_output["safety"]["emotion_label"],
        "safety_flag": pipeline_output["safety"]["safety_flag"],
        "triggered_by": pipeline_output["safety"]["triggered_by"]
    },
    "knowledge": {
        "knowledge_found": pipeline_output["knowledge"]["knowledge_found"],
        "sources": pipeline_output["knowledge"]["sources"],
        "retrieval_skipped": pipeline_output["knowledge"]["retrieval_skipped"]
    },
    "persona_prompt_input_preview": pipeline_output["persona"]["prompt_input"][:350]}, indent=2, ensure_ascii=False))

{
  "safety": {
    "emotion_label": "Suicidal",
    "safety_flag": "crisis",
    "triggered_by": "model"
  },
  "knowledge": {
    "knowledge_found": false,
    "sources": [],
    "retrieval_skipped": true
  },
  "persona_prompt_input_preview": "I want to hang myself"
}
